# MLOps Pipeline for Hospital Readmission Classification using the UCI Diabetes Dataset

This project implements an end-to-end MLOps pipeline for predicting hospital readmission risk using the UCI Diabetes 130-US Hospitals dataset. The objective is to build a reliable machine learning classification system that identifies whether a patient is likely to be readmitted within 30 days based on historical clinical records.

The dataset contains over 100,000 patient records with 47 features, including demographic information, diagnoses, medication history, and hospital visit details. It presents real-world challenges such as missing values in multiple formats, high-cardinality categorical variables, and significant class imbalance.

The pipeline covers data ingestion, preprocessing, feature engineering, model training, and evaluation using appropriate metrics such as ROC-AUC, precision, and recall. The project is structured with an MLOps mindset to support reproducibility, scalability, and deployment readiness in a healthcare environment.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv("diabetic_data.csv", na_values = "?")
data.sample(10)

C:\Users\HomePC\AppData\Local\Temp\ipykernel_34012\1229332509.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("diabetic_data.csv", na_values = "?")


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
38029,118089780,23227992,AfricanAmerican,Female,[70-80),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
56265,161697720,76476285,Caucasian,Male,[50-60),[50-75),3,1,1,1,...,No,No,No,No,No,No,No,No,No,NO
36082,111199554,112997997,Caucasian,Female,[60-70),NaN,3,3,1,7,...,No,No,No,No,No,No,No,No,Yes,NO
20694,72744468,13023792,Caucasian,Male,[30-40),NaN,1,1,7,4,...,No,Steady,No,No,No,No,No,No,Yes,<30
42431,130857570,23688702,Caucasian,Male,[70-80),NaN,3,1,1,7,...,No,Steady,No,No,No,No,No,No,Yes,>30
58889,166305330,77400189,Caucasian,Male,[70-80),NaN,1,1,7,9,...,No,No,No,No,No,No,No,No,No,>30
74288,220978230,86390334,Caucasian,Male,[70-80),[100-125),1,1,7,4,...,No,No,No,No,No,No,No,Ch,Yes,<30
5395,28235370,15586650,Caucasian,Female,[70-80),NaN,1,1,7,1,...,No,No,No,No,No,No,No,No,Yes,NO
58710,166069038,85378401,Caucasian,Female,[80-90),NaN,1,6,1,5,...,No,No,No,No,No,No,No,No,Yes,<30
64634,179900544,35580915,AfricanAmerican,Male,[50-60),NaN,1,6,7,10,...,No,Down,No,No,No,No,No,Ch,Yes,NO


In [3]:
data.shape

(101766, 50)

In [4]:
missing = data.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(data)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_percent
})

missing_df.head(10)

,missing_count,missing_percent
weight,98569,96.858479
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
medical_specialty,49949,49.082208
payer_code,40256,39.557416
race,2273,2.233555
diag_3,1423,1.398306
diag_2,358,0.351787
diag_1,21,0.020636
encounter_id,0,0.000000


The dataset contains 101,766 patient encounters and 50 features. It is a real-world clinical dataset characterized by significant missing data, particularly in laboratory and specialty-related features. The level of missingness varies across features, requiring a combination of feature removal and targeted imputation strategies to ensure data quality for modeling.

Columns with extremely high levels of missing data (typically above 80%) were dropped from the dataset as part of the data preprocessing stage. These features lacked sufficient information density to support reliable statistical imputation and would likely introduce noise or bias if retained. Removing them helps improve data quality, reduces dimensionality, and ensures the machine learning model is trained on more meaningful and informative features.

In [5]:
data.drop(columns=['weight', 'max_glu_serum', 'A1Cresult'], inplace=True)


data['medical_specialty'] = data['medical_specialty'].fillna('missing')
data['payer_code'] = data['payer_code'].fillna('missing')

threshold = 0.01

top_specialties = data['medical_specialty'].value_counts(normalize=True)
top_specialties = top_specialties[top_specialties > threshold].index
data['medical_specialty'] = data['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'other'
)

top_payers = data['payer_code'].value_counts(normalize=True)
top_payers = top_payers[top_payers > threshold].index
data['payer_code'] = data['payer_code'].apply(
    lambda x: x if x in top_payers else 'other'
)

In [6]:
data[['medical_specialty', 'payer_code']].isnull().sum()

medical_specialty    0
payer_code           0
dtype: int64

In [7]:
data['medical_specialty'].value_counts().head(10)

medical_specialty
missing                       49949
InternalMedicine              14635
other                          8340
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Name: count, dtype: int64

In [8]:
data.duplicated().sum()

np.int64(0)

In [9]:
data['readmitted'].value_counts(normalize=True)

readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

The dataset was checked for duplicate records and none were found, confirming data integrity at the row level.

The target variable (readmitted) shows a moderate class imbalance:
- NO: 53.9%
- >30: 34.9%
- <30: 11.2%

This indicates a 3-class classification problem with a relatively smaller minority class (<30 days readmission), which will require careful model evaluation using metrics such as F1-score and recall rather than accuracy alone.

Missing values were consistently encoded using the label 'missing' across categorical features to ensure uniform representation of incomplete data during model training.

In [10]:
data['race'] = data['race'].fillna('missing')
for col in ['diag_1', 'diag_2', 'diag_3']:
    data[col] = data[col].fillna('missing')

In [11]:
data['race'] = data['race'].replace({'AfricanAmerican': 'African American'})

In [12]:
missing = data.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(data)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_percent
})

missing_df.head(10)

,missing_count,missing_percent
encounter_id,0,0.0
tolazamide,0,0.0
acetohexamide,0,0.0
glipizide,0,0.0
glyburide,0,0.0
tolbutamide,0,0.0
pioglitazone,0,0.0
rosiglitazone,0,0.0
acarbose,0,0.0
miglitol,0,0.0


In [13]:
data.isnull().sum().sum()

np.int64(0)

In [14]:
data.duplicated().sum()

np.int64(0)

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 47 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   admission_type_id         101766 non-null  int64 
 6   discharge_disposition_id  101766 non-null  int64 
 7   admission_source_id       101766 non-null  int64 
 8   time_in_hospital          101766 non-null  int64 
 9   payer_code                101766 non-null  object
 10  medical_specialty         101766 non-null  object
 11  num_lab_procedures        101766 non-null  int64 
 12  num_procedures            101766 non-null  int64 
 13  num_medications           101766 non-null  int64 
 14  numb

In [16]:
data.select_dtypes(include='object').nunique().sort_values()

citoglipton                   1
examide                       1
tolbutamide                   2
change                        2
metformin-pioglitazone        2
metformin-rosiglitazone       2
glimepiride-pioglitazone      2
glipizide-metformin           2
troglitazone                  2
diabetesMed                   2
acetohexamide                 2
tolazamide                    3
readmitted                    3
gender                        3
chlorpropamide                4
glipizide                     4
glyburide                     4
nateglinide                   4
pioglitazone                  4
rosiglitazone                 4
acarbose                      4
miglitol                      4
repaglinide                   4
metformin                     4
glimepiride                   4
insulin                       4
glyburide-metformin           4
race                          6
age                          10
medical_specialty            11
payer_code                   11
diag_1  

In [17]:
np.isinf(data.select_dtypes(include=['number'])).sum()

encounter_id                0
patient_nbr                 0
admission_type_id           0
discharge_disposition_id    0
admission_source_id         0
time_in_hospital            0
num_lab_procedures          0
num_procedures              0
num_medications             0
number_outpatient           0
number_emergency            0
number_inpatient            0
number_diagnoses            0
dtype: int64

In [18]:
data.describe(include='all')

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
count,1.017660e+05,1.017660e+05,101766,101766,101766,101766.000000,101766.000000,101766.000000,101766.000000,101766,...,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766
unique,NaN,NaN,6,3,10,NaN,NaN,NaN,NaN,11,...,1,4,4,2,2,2,2,2,2,3
top,NaN,NaN,Caucasian,Female,[70-80),NaN,NaN,NaN,NaN,missing,...,No,No,No,No,No,No,No,No,Yes,NO
freq,NaN,NaN,76099,54708,26068,NaN,NaN,NaN,NaN,40256,...,101766,47383,101060,101753,101765,101764,101765,54755,78363,54864
mean,1.652016e+08,5.433040e+07,NaN,NaN,NaN,2.024006,3.715642,5.754437,4.395987,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.026403e+08,3.869636e+07,NaN,NaN,NaN,1.445403,5.280166,4.064081,2.985108,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.252200e+04,1.350000e+02,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,8.496119e+07,2.341322e+07,NaN,NaN,NaN,1.000000,1.000000,1.000000,2.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.523890e+08,4.550514e+07,NaN,NaN,NaN,1.000000,1.000000,7.000000,4.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2.302709e+08,8.754595e+07,NaN,NaN,NaN,3.000000,4.000000,7.000000,6.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
data['readmitted'].value_counts(normalize=True)

readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

In [20]:
data = pd.read_csv("IDS_mapping.csv")
data.info()
data.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   admission_type_id  65 non-null     object
 1   description        62 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


,admission_type_id,description
32,23,Discharged/transferred to a long term care hos...
22,13,Hospice / home
33,24,Discharged/transferred to a nursing facility c...
31,22,Discharged/transferred to another rehab fac in...
2,3,Elective
0,1,Emergency
3,4,Newborn
24,15,Discharged/transferred within this institution...
41,admission_source_id,description
30,21,"Expired, place unknown. Medicaid only, hospice."
